# Chapter 10: Real-time Gaming Leaderboard
*System Design Interview Volume 2*

## TL;DR

A real-time gaming leaderboard ranks millions of players by score with sub-second updates. **SQL ranking** (ORDER BY + row scan) is O(n) per query and collapses at scale. **Redis sorted sets** -- backed by a hash table + skip list -- deliver O(log n) rank lookups, range queries, and score updates. At 5M DAU, a single Redis node suffices (around 650 MB). At 500M DAU, **fixed-partition sharding** by score range is preferred over hash partitioning because it supports exact rank computation. A **DynamoDB alternative** uses write-sharded global secondary indexes with scatter-gather for top-K, but can only provide approximate (percentile-based) ranking. MySQL backs Redis for durability and disaster recovery.

## Requirements

| Type | Requirement | Detail |
|---|---|---|
| Functional | Display top 10 | Show the top 10 players on the leaderboard |
| Functional | Show user rank | Return a specific user's rank in real time |
| Functional | Relative position | Show players 4 places above and below a user |
| Non-functional | Real-time updates | Score changes reflected immediately |
| Non-functional | Scalability | Support 5M DAU (extensible to 500M) |
| Non-functional | Availability | Highly available with failover strategy |

## Estimation: QPS and Storage

In [ ]:
# Back-of-envelope for gaming leaderboard
dau = 5_000_000
mau = 25_000_000
games_per_day = 10

# QPS calculations
seconds_per_day = 86400
avg_concurrent = dau / seconds_per_day
peak_multiplier = 5

print("=== QPS Estimates ===")
print(f"Average concurrent users: {avg_concurrent:.0f}")
print(f"Peak concurrent users: {avg_concurrent * peak_multiplier:.0f}")

# Score updates: each game can result in a win
score_update_qps_avg = avg_concurrent * games_per_day
score_update_qps_peak = score_update_qps_avg * peak_multiplier
print(f"\nScore update QPS (avg): {score_update_qps_avg:.0f}")
print(f"Score update QPS (peak): {score_update_qps_peak:.0f}")

# Leaderboard reads: once per user per day
leaderboard_read_qps = avg_concurrent
print(f"Leaderboard read QPS: {leaderboard_read_qps:.0f}")

# Storage: Redis sorted set
user_id_bytes = 24
score_bytes = 2
entry_bytes = user_id_bytes + score_bytes
# Worst case: all MAU have at least one entry
raw_storage_mb = mau * entry_bytes / (1024 * 1024)
# 2x overhead for skip list + hash internals
total_storage_mb = raw_storage_mb * 2
print(f"\n=== Storage Estimates ===")
print(f"Raw leaderboard storage: {raw_storage_mb:.0f} MB")
print(f"With 2x skip-list overhead: {total_storage_mb:.0f} MB")
print(f"Fits on a single Redis node: {total_storage_mb < 16_000}")

# Scale to 500M DAU
scale_factor = 100
scaled_storage_gb = total_storage_mb * scale_factor / 1024
scaled_qps = score_update_qps_peak * scale_factor
print(f"\n=== At 500M DAU (100x) ===")
print(f"Leaderboard storage: {scaled_storage_gb:.0f} GB")
print(f"Peak score update QPS: {scaled_qps:,.0f}")
print(f"Requires sharding: {scaled_storage_gb > 16}")

## High-Level Design

```mermaid
graph TB
    Player([Player])

    subgraph GameFlow["Game Flow"]
        GS[Game Service]
    end

    subgraph LeaderboardFlow["Leaderboard Service"]
        LS[Leaderboard API]
    end

    subgraph Storage["Storage Layer"]
        REDIS[(Redis Sorted Set)]
        MYSQL[(MySQL<br/>User + Point tables)]
        CACHE[User Profile Cache]
    end

    Player -->|1. Win game| GS
    GS -->|2. Update score| LS
    LS -->|3. ZINCRBY| REDIS
    LS -->|4. Insert point| MYSQL
    Player -->|5. View leaderboard| LS
    LS -->|6. ZREVRANGE / ZREVRANK| REDIS
    LS -->|7. Fetch user details| CACHE
    CACHE -.->|miss| MYSQL
```

## Deep Dive

### Why SQL Ranking Fails at Scale

A relational approach uses `ORDER BY score DESC` with row numbering. This requires a **full table scan** to determine any user's rank -- O(n log n) per query. With millions of rows, queries take 10+ seconds. The data changes constantly, so caching is not feasible. Index optimizations help for top-K but not for arbitrary rank lookups.

### Redis Sorted Sets and Skip Lists

```mermaid
graph TB
    subgraph SortedSet["Redis Sorted Set Internals"]
        HT["Hash Table<br/>member -> score<br/>O(1) lookup"]
        SL["Skip List<br/>score -> member<br/>O(log n) range/rank"]
    end

    subgraph Operations["Key Operations"]
        ZADD["ZADD / ZINCRBY<br/>O(log n)"]
        ZREV["ZREVRANGE<br/>O(log n + m)"]
        ZRANK["ZREVRANK<br/>O(log n)"]
    end

    HT <--> SL
    ZADD --> SL
    ZREV --> SL
    ZRANK --> SL
```

**Skip list**: a probabilistic data structure with multi-level indexes over a sorted linked list. Each level skips roughly half the nodes of the level below. A 64-element list with 5 index levels needs only 11 node traversals instead of 62.

**Core Redis commands for leaderboards:**

| Command | Purpose | Example | Complexity |
|---|---|---|---|
| ZINCRBY | Add score on win | `ZINCRBY leaderboard_feb 1 'mary'` | O(log n) |
| ZREVRANGE | Top K players | `ZREVRANGE leaderboard_feb 0 9 WITHSCORES` | O(log n + k) |
| ZREVRANK | User's rank | `ZREVRANK leaderboard_feb 'mary'` | O(log n) |
| ZREVRANGE | Neighbors | `ZREVRANGE leaderboard_feb 357 365` | O(log n + m) |

### Leaderboard Architecture Options

**Self-managed deployment:**
- Redis sorted set for leaderboard data
- MySQL for user details and point history (disaster recovery source)
- Optional user profile cache for top 10 players

**Serverless (AWS) deployment:**
- API Gateway routes to Lambda functions
- Lambda calls Redis (ElastiCache) and MySQL (RDS)
- Auto-scales with DAU growth; no server management

### Sharding at 500M DAU

```mermaid
graph TB
    subgraph FixedPartition["Fixed Partition Sharding"]
        S1["Shard 1<br/>scores 1-100"]
        S2["Shard 2<br/>scores 101-200"]
        S3["..."]
        S10["Shard 10<br/>scores 901-1000"]
    end

    subgraph HashPartition["Hash Partition (Redis Cluster)"]
        H1["Shard 1<br/>slots 0-5500"]
        H2["Shard 2<br/>slots 5501-11000"]
        H3["Shard 3<br/>slots 11001-16383"]
    end

    APP[Application] -->|score range lookup| FixedPartition
    APP -->|CRC16 mod 16384| HashPartition
```

**Fixed partition** (preferred):
- Shards by score range (e.g., 10 shards, each covering 100 points)
- Top-K fetched from the highest-score shard only
- User rank = local rank + sum of counts from all higher shards
- `INFO KEYSPACE` returns shard size in O(1)
- May need rebalancing if score distribution is uneven

**Hash partition** (Redis Cluster):
- CRC16(key) % 16384 assigns slots automatically
- Top-K requires scatter-gather across all shards
- Cannot easily compute exact user rank
- Higher latency for top-K with many shards

### NoSQL Alternative: DynamoDB

- Global secondary index: partition key = `game#year-month#partition_number`, sort key = score
- **Write sharding** appends `user_id % n` to partition key to avoid hot partitions
- Top-K via scatter-gather across n partitions
- Exact rank is impractical; percentile ranking (e.g., "top 10-20%") via periodic cron analysis of score distributions

### Failure Recovery

Redis persistence (RDB/AOF) plus read replicas. On main node failure, promote a replica. For catastrophic failure, replay the MySQL point table entries with ZINCRBY to rebuild the sorted set offline.

### Tie-Breaking

Use a Redis Hash to store user_id to timestamp of most recent win. When two players have the same score, rank the one with the **older** timestamp higher (they achieved the score first).

## Trade-offs

| Dimension | Option A | Option B |
|---|---|---|
| Data store | Redis sorted set: O(log n) ops, in-memory | SQL: familiar, durable, but O(n) ranking |
| Deployment | Self-managed: full control | Serverless (Lambda): auto-scale, less ops |
| Sharding | Fixed partition: exact rank, uneven load | Hash partition: even load, no exact rank |
| Persistence | Redis + MySQL backup: fast + recoverable | DynamoDB: fully managed, no Redis needed |
| Rank precision | Exact rank: complex at scale | Percentile rank: simple, good UX at scale |

## Key Takeaways

1. **Redis sorted sets** are the go-to for real-time leaderboards -- O(log n) for all key operations
2. **Skip lists** make sorted sets fast: multi-level indexes turn O(n) scans into O(log n) traversals
3. **Server-side score updates only** -- never let the client set scores directly (security)
4. **Fixed-partition sharding** preserves exact rank computation at extreme scale
5. **MySQL as recovery source** -- every point is recorded with a timestamp for full leaderboard rebuild
6. **Monthly rotation** of sorted sets keeps working-set size bounded

## Related Concepts

- [[redis-sorted-sets]] -- data structure powering the leaderboard
- [[skip-list]] -- probabilistic structure behind sorted set performance
- [[leaderboard-architecture]] -- game service, leaderboard service, storage layer
- [[leaderboard-sharding]] -- fixed vs hash partitioning at scale
- [[nosql-leaderboard]] -- DynamoDB alternative with write sharding
- [[sql-vs-redis-ranking]] -- why SQL ORDER BY collapses at millions of rows